# 階段 3：微調 Chronos-2（Colab Enterprise）

用 12 站 2021–2023 年資料微調**一個**全域 Chronos-2，以 2024 年選學習率與最佳檢查點，再對 2025 年做與其他模型完全相同的滾動評估。共三個變體：單變量、加過去共變數、加 JMA GSM 預報風場。

1. 連線到 **L4 GPU** 執行階段，把新的 `pm25_project.zip` 上傳到左側 📁 檔案瀏覽器（覆蓋舊檔）。
2. 按「全部執行」。預估總共 1.5–3 小時（3 個變體 × 2 個學習率的微調，加上 2025 年推論）。
3. **中途斷線可以續跑**：重新執行全部即可，已完成的微調與預測會自動跳過。
4. 任何一格出錯都會停下，請截圖或貼最後 30 行給 Claude。
5. 最後一格會下載 `pm25_finetune_results.zip`（只有預測檔與摘要，不含 1.4 GB 的模型權重）。
6. **跑完務必停止執行階段**。

In [ ]:
%%bash
set -e
# 1. 檢查 GPU 與 Python 版本
nvidia-smi || echo '沒有 GPU，將使用 CPU（較慢但可跑）'
python --version

In [ ]:
# 2. 解壓縮專案（自動尋找上傳的 zip；Colab Enterprise 的檔案瀏覽器根目錄是 /）
import os
candidates = ['/pm25_project.zip', '/content/pm25_project.zip',
              os.path.join(os.getcwd(), 'pm25_project.zip')]
found = [p for p in candidates if os.path.exists(p)]
assert found, '找不到 pm25_project.zip，請先上傳到左側檔案瀏覽器'
os.chdir(os.path.dirname(found[0]))
!rm -rf pm25/src pm25/data/processed
!unzip -q -o pm25_project.zip -d pm25
os.chdir('pm25')
print(os.getcwd())
!ls data/processed | head -3

In [ ]:
%%bash
set -e
# 3. 安裝套件（每次上傳新 zip 後都要重跑；numba 的 numpy 版本警告可忽略）
echo "目前目錄：$(pwd)"
test -f pyproject.toml || { echo "目錄不對：請先重跑第 2 格"; exit 1; }
pip uninstall -q -y pm25-tsfm || true
pip install -q '.[models]'
pip install -q --force-reinstall --no-deps .
cd / && python -c "import pandas, torch; from pm25.models.device import default_device; print('pandas', pandas.__version__, '| torch', torch.__version__, '| device =', default_device())"

## 微調（GPU）

In [ ]:
%%bash
set -e
# 4. 三個變體 × 兩個學習率的全參數微調；每次結束印出驗證損失摘要
python -m pm25.models.finetune

In [ ]:
%%bash
set -e
# 5. 微調摘要（選中的學習率與 2024 驗證損失）
cat results/finetuned/*.json

## 2025 年滾動評估

In [ ]:
%%bash
set -e
# 6. 用微調後的模型預測 2025 年每小時起點
python -m pm25.experiment --family finetuned

## 打包與下載

In [ ]:
# 7. 確認 36 個預測檔（12 站 × 3 變體）後打包下載
import glob
models = ['chronos2_ft_uni', 'chronos2_ft_cov', 'chronos2_ft_nwp']
files = [f for m in models for f in glob.glob(f'results/predictions/*__{m}.csv.gz')]
files += glob.glob('results/finetuned/*.json')
print('檔案數量：', len(files), '/ 39')
assert len(files) == 39, '檔案不齊全，請往上檢查哪一格出錯'
!rm -f ../pm25_finetune_results.zip
!zip -q ../pm25_finetune_results.zip {' '.join(files)}
!ls -lh ../pm25_finetune_results.zip
from google.colab import files as colab_files
colab_files.download('../pm25_finetune_results.zip')

## 結束
到 Colab Enterprise →「執行階段」把這個執行階段**停止或刪除**，避免繼續計費。